# Multi Whisker-Pole Contact Classifier / Collision — Inference

Run the trained EfficientNet-B3 on a video and produce:
1. **Per-frame CSV** — every frame with its contact prediction (0/1) and probability
2. **Interval CSV** — contiguous contact regions in `Start,End` format

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

INFERENCE_DIR = os.path.dirname(os.path.abspath("__file__"))
if INFERENCE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_DIR)

from utils import (
    load_model,
    get_inference_transforms,
    VideoFrameDataset,
    run_inference,
    frames_to_intervals,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1 — Configuration

In [ ]:
# ========================  PATHS  ========================

VIDEO_PATH = "/home/mvdokh/data/052725_1/052725_piezo_1.mp4"


CHECKPOINT_PATH = os.path.join(
    os.path.dirname(INFERENCE_DIR),
    "Training", "checkpoints", "contact_classifier.pt"
)

# Output CSVs will be saved here
OUTPUT_DIR = os.path.join(INFERENCE_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================  SETTINGS  ========================

IMG_SIZE      = 256
BATCH_SIZE    = 64       # can be larger than training since no gradients
NUM_WORKERS   = 2        # try higher to parallelize video decoding
THRESHOLD     = 0.5      # probability threshold for contact
START_FRAME   = 0
END_FRAME     = 250_000  # only first 250k frames (second half has no pole)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
Checkpoint: /orcd/home/002/mvdokh/Deep-Learning/Collision Classification/Training/checkpoints/contact_classifier.pt
Output dir: /orcd/home/002/mvdokh/Deep-Learning/Collision Classification/Inference/results


## 2 — Load Model

In [ ]:
model = load_model(CHECKPOINT_PATH, DEVICE)

Loaded checkpoint from: /orcd/home/002/mvdokh/Deep-Learning/Collision Classification/Training/checkpoints/contact_classifier.pt
  Config: {'img_size': 256, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'freeze_backbone': False, 'model_arch': 'efficientnet_b3'}
  Epoch: 13
  Val loss: 0.0047


## 3 — Create Video Dataset & DataLoader

In [ ]:
transform = get_inference_transforms(IMG_SIZE)

dataset = VideoFrameDataset(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    transform=transform,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Total frames to process: {len(dataset):,}")
print(f"Batches: {len(dataloader):,}")

VideoFrameDataset: frames 0–249999 (250,000 frames)
Total frames to process: 250,000
Batches: 3,907


## 4 — Run Inference

In [ ]:
results_df = run_inference(model, dataloader, DEVICE, threshold=THRESHOLD)

n_contact = (results_df["contact"] == 1).sum()
n_no_contact = (results_df["contact"] == 0).sum()
print(f"\nTotal frames   : {len(results_df):,}")
print(f"Contact frames : {n_contact:,} ({100*n_contact/len(results_df):.1f}%)")
print(f"No contact     : {n_no_contact:,} ({100*n_no_contact/len(results_df):.1f}%)")
print()
results_df.head(10)

Inference: 100%|██████████| 3907/3907 [11:57<00:00,  5.44it/s]



Total frames   : 250,000
Contact frames : 8,169 (3.3%)
No contact     : 241,831 (96.7%)



,frame,probability,contact
0,0,2.009479e-09,0
1,1,2.032902e-09,0
2,2,1.363553e-09,0
3,3,5.814693e-10,0
4,4,4.039175e-10,0
5,5,1.510969e-09,0
6,6,1.723433e-09,0
7,7,3.592149e-09,0
8,8,1.944357e-09,0
9,9,1.695847e-09,0


## 5 — Convert to Intervals & Save CSVs

In [ ]:
# Per-frame CSV
per_frame_path = os.path.join(OUTPUT_DIR, "per_frame_predictions.csv")
results_df.to_csv(per_frame_path, index=False)
print(f"Saved per-frame predictions → {per_frame_path}")

# Contact intervals CSV
intervals_df = frames_to_intervals(results_df, label_col="contact", label_val=1)
intervals_path = os.path.join(OUTPUT_DIR, "collision_intervals.csv")
intervals_df.to_csv(intervals_path, index=False)
print(f"Saved contact intervals     → {intervals_path}")
print(f"\nFound {len(intervals_df)} contact intervals")
print()
intervals_df.head(20)

Saved per-frame predictions → /orcd/home/002/mvdokh/Deep-Learning/Collision Classification/Inference/results/per_frame_predictions.csv
Saved contact intervals     → /orcd/home/002/mvdokh/Deep-Learning/Collision Classification/Inference/results/collision_intervals.csv

Found 435 contact intervals



,Start,End
0,7110,7110
1,7201,7216
2,34528,34533
3,34756,34758
4,34765,34767
5,34939,34945
6,34982,34985
7,34987,34989
8,35422,35427
9,35611,35611


## 6 — Quick Summary

In [ ]:
# Interval length stats
intervals_df["length"] = intervals_df["End"] - intervals_df["Start"] + 1

print(f"Contact intervals: {len(intervals_df)}")
print(f"Total contact frames: {intervals_df['length'].sum():,}")
print(f"Avg interval length: {intervals_df['length'].mean():.1f} frames")
print(f"Min interval length: {intervals_df['length'].min()} frames")
print(f"Max interval length: {intervals_df['length'].max()} frames")
print(f"Median interval length: {intervals_df['length'].median():.0f} frames")

Contact intervals: 435
Total contact frames: 8,169
Avg interval length: 18.8 frames
Min interval length: 1 frames
Max interval length: 994 frames
Median interval length: 7 frames
